In [2]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import mlflow

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model loaded successfully


In [6]:
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")

print(dataset)
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}


In [7]:
label_map = {0: "bearish", 1: "bullish", 2: "neutral"}

def format_prompt(example):
    text = example["text"]
    sentiment = label_map[example["label"]]
    return {
        "text": f"<|im_start|>user\nAnalyze the sentiment of this financial news: {text}<|im_end|>\n<|im_start|>assistant\nThe sentiment is {sentiment}.<|im_end|>"
    }

formatted = dataset.map(format_prompt)
print(formatted["train"][0]["text"])

Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

<|im_start|>user
Analyze the sentiment of this financial news: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT<|im_end|>
<|im_start|>assistant
The sentiment is bearish.<|im_end|>


In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 1,714,522,112 || trainable%: 0.1835


In [17]:
mlflow.set_experiment("financial-sentiment-finetuning")

training_args = SFTConfig(
    output_dir="./smollm2-financial",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=False, # Disable FP16 to avoid BFloat16 errors
    bf16=False, # Explicitly disable BFloat16
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    max_length=256,
    dataset_text_field="text",
    report_to="none",
)

with mlflow.start_run(run_name="smollm2-lora-run1"):
    mlflow.log_params({
        "model": "SmolLM2-1.7B-Instruct",
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 2,
        "learning_rate": 2e-4,
        "dataset": "twitter-financial-news-sentiment"
    })

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=formatted["train"],
        eval_dataset=formatted["validation"],
        peft_config=lora_config
    )

    trainer.train()
    mlflow.log_metric("final_train_loss", trainer.state.log_history[-1].get("train_loss", 0))


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,1.728992,1.708815
2,1.680923,1.692110


In [18]:
trainer.save_model("./smollm2-financial-finetuned")
mlflow.log_metric("final_train_loss", 1.680923)
mlflow.log_metric("final_eval_loss", 1.692110)
print("Model saved and metrics logged!")

Model saved and metrics logged!
